# DRL Portfolio Management — Results

Deep RL agents (PPO, A2C, DDPG) for portfolio allocation, following Jiang et al. (2017)
and compared against Liang et al. (2018).

Every table and figure below is **regenerated live in this notebook** from the trained
checkpoints — nothing is pre-rendered, and nothing is read from private storage.

**To run it yourself:** Cell 1 (install) → *Runtime → Restart session* → *Run all*.
Everything else is automatic; the code, data and checkpoints come from the public
repository, so there is nothing to upload.

Source: [github.com/FemiOje/thesis-drl-portfolio](https://github.com/FemiOje/thesis-drl-portfolio)

## 1. Install dependencies

Colab ships numpy 2.x; stable-baselines3 2.9 and this codebase require numpy < 2.

In [ ]:
!pip install -q "stable-baselines3==2.9.0" "gymnasium==1.0.0" "numpy<2.0" yfinance
print("Installed. Now: Runtime -> Restart session, then continue from Cell 2.")

## 2. Fetch the project

Clones the repository, which carries the code, the raw price CSVs, and the 12 trained
checkpoints. Nothing to upload and nothing read from private storage — re-running this
cell is safe.

In [ ]:
!git clone -q https://github.com/FemiOje/thesis-drl-portfolio.git 2>/dev/null || echo "already cloned"
%cd /content/thesis-drl-portfolio
!git pull -q

### Verify the environment before spending any compute

In [ ]:
import glob, numpy, subprocess

print("numpy:", numpy.__version__, "(must be 1.x — restart the session if it is 2.x)")
print("commit:", subprocess.run(["git", "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())
print("price CSVs:  ", len(glob.glob("data/raw/*.csv")), "(expect 8)")
print("checkpoints: ", len(glob.glob("results/models_conv/*/*/best_model.zip")), "(expect 12)")

## 3. Sanity check

Seven unit tests cover the environment against Jiang's equations — the commission model,
weight drift, the reward, and the round-trip invariants.

In [ ]:
!python -m pytest tests/ -q

## 4. Evaluate all three algorithms on the held-out test window

Primary sweep: weight-shared convolutional encoder, 500k steps, action bound ±5.
PPO and A2C have 5 seeds; DDPG has 2 — it froze identically on both, see §6.
Missing seeds are skipped with a warning, which is expected for DDPG.

In [ ]:
!python experiments/evaluate.py --split test \
    --algos ppo a2c ddpg --seeds 0 1 2 3 4 \
    --models-dir results/models_conv \
    --out-dir results/evaluation_conv/test \
    --action-bound 5

### Results table (rendered inline)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/evaluation_conv/test/RESULTS.md").read()))

### Per-seed detail

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
display(pd.read_csv("results/evaluation_conv/test/per_seed_results.csv"))

### Figures

In [ ]:
from IPython.display import Image, display

for f in ["01_wealth_curves.png", "02_allocation_heatmaps.png", "03_seed_distributions.png"]:
    display(Image(filename="results/evaluation_conv/test/" + f))

## 5. Before / after: what the feature extractor changed

SB3's `MultiInputPolicy` gives a Dict entry a CNN **only if it is an image space**. The
`(3, 50, 8)` price tensor is not one, so it took the `nn.Flatten()` branch — 1200 unrelated
inputs, and no convolution anywhere in the trained models. Replacing that with a
weight-shared convolutional encoder (`experiments/extractors.py`, kernels spanning
*k timesteps × 1 asset* — Jiang's Identical Independent Evaluators property) is the single
intervention separating the two columns below.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

comparison = pd.DataFrame(
    {
        "Ablation (flatten, 300k, +/-10)": ["1.1507 +/- 0.0108", "1.1033 +/- 0.1532",
                                            "1.1450 +/- 0.0603", "1.1506", "10 of 10"],
        "Primary (conv, 500k, +/-5)":      ["1.2074 +/- 0.0893", "1.1982 +/- 0.0341",
                                            "1.1720 (n=2)",      "1.1506", "0 of 10"],
    },
    index=["PPO test fAPV", "A2C test fAPV", "DDPG test fAPV",
           "Buy & Hold", "Frozen policies (PG methods)"],
)
display(comparison)

display(Markdown("**Ablation results in full, for reference:**"))
display(Markdown(open("results/evaluation/test/RESULTS.md").read()))

## 6. Policy diagnostic — does each policy actually respond to the market?

fAPV cannot distinguish a trading policy from a constant one that happened to do well.
This probes every checkpoint on observations from **nine widely separated market regimes**
while holding `w_prev` fixed, so any change in output is attributable to the price tensor
alone.

| Verdict | Meaning |
|---|---|
| `FROZEN` | weights never move |
| `MARKET-BLIND` | weights move, but from `w_prev` feedback, not from the prices |
| `negligible` | responds by under 1 percentage point across regimes |
| `responds to X_t` | genuine trading policy |

In [ ]:
!python experiments/policy_diagnostic.py \
    --algos ppo a2c ddpg --seeds 0 1 2 3 4 \
    --models-dir results/models_conv \
    --split test --action-bound 5

**Reading of this output.** 4 of 12 policies respond meaningfully — all of them PPO.

* **The encoder eliminated freezing** for the policy-gradient methods: 0 of 10, against
  10 of 10 in the ablation. This is the causal evidence for the architecture claim.
* **A2C did not benefit.** Every seed is market-blind or negligible despite receiving the
  identical encoder. Its 1.1982 is a constant portfolio that happened to perform well, and
  must not be reported as a working agent.
* **DDPG is frozen on both seeds.** Its tanh-squashed actor is saturated — 100% of output
  dimensions sit exactly on the bound. Saturated tanh has zero gradient, so the policy can
  never update again. Reproduced across two action bounds (±10, ±5) and two extractors:
  the bound was the blast radius, not the cause.
* **Turnover corroborates the probe independently.** PPO's responsive seeds trade at
  0.083–0.156 per step; its market-blind seed 3 trades at 0.026.

## 7. Training curves

Validation fAPV against timesteps, mean ± std across seeds.

In [ ]:
!python experiments/plot_training.py --models-dir results/models_conv \
    --out-dir results/evaluation_conv

import os
from IPython.display import Image, display

p = "results/evaluation_conv/training_curves.png"
display(Image(filename=p)) if os.path.exists(p) else print("curve figure not generated")

## 8. What can and cannot be claimed

**Supported by the evidence above**

1. **Architecture dominated algorithm choice.** Under the default extractor all three
   algorithms were statistically indistinguishable *because all were constant policies*.
   Changing only the encoder lifted every algorithm above the passive benchmarks on return.
   No other change in this project came close.
2. **All agents beat the benchmarks on return; none on risk.** PPO returns 29.0% annualized
   against Buy & Hold's 20.8%, but Buy & Hold's Sharpe of 1.852 beats every agent, and its
   5.64% drawdown is under half PPO's 12.30%. The objective has no risk term, so the agents
   optimized exactly what they were asked to.
3. **DDPG's failure mechanism is identified,** not merely observed: actor saturation against
   the action bounds, reproduced four times across two bounds and two extractors.

**Not supported — state these before anyone asks**

* **No PPO-vs-A2C ranking is defensible.** 1.2074 vs 1.1982 is a 0.009 gap against a PPO
  standard deviation of 0.089 at n=5, and two of five PPO seeds fall below Buy & Hold.
* **PPO is not reliable either** — 1 of its 5 seeds is market-blind. The result is a success
  *rate* (4/5 vs 0/5 vs 0/2), not a categorical difference between algorithms.
* **A2C ran at `n_steps=16` against PPO's 2048** — a 128× batch gap that was a project
  choice, not an SB3 default (SB3's A2C default is 5). Until that is equalized, A2C's
  failure cannot be cleanly attributed to the algorithm rather than to the batch size.
* **A2C had not converged at 500k** (+0.0273 fAPV per 100k, still rising).
* **DDPG is n=2**, and no mean ± std claim is made from it.

## 9. Next experiment

Run A2C at PPO's batch size. If A2C starts responding, the cause is batch-size
signal-to-noise; if it stays blind, the cause is the missing trust region. A2C at PPO's
batch size is essentially *PPO without the clip*, so this isolates one variable rather than
merely retuning.

Roughly 2 hours per seed on CPU. Uncomment to run.

In [ ]:
# !python experiments/train.py --algo a2c --seeds 0 1 2 --steps 500000 \
#     --extractor conv --action-bound 5 --n-steps 2048 --lr 3e-4 \
#     --models-dir results/models_a2c_bigbatch --device cpu